In [ ]:
#@title 📦 **نصب و راه‌اندازی**
#@markdown این مرحله حدود 3 دقیقه طول می‌کشد

from IPython.display import clear_output
import subprocess
import os

colab_path = "/content"
kaggle_path = "/kaggle/working"

if os.path.exists(colab_path):
    print("🔵 در حال اتصال به Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    path = "/content"
    print("✅ اتصال برقرار شد")
elif os.path.exists(kaggle_path):
    print("⚠️ محیط Kaggle شناسایی شد")
    path = "/kaggle/working"
else:
    raise EnvironmentError("❌ خطا: محیط اجرا شناسایی نشد")

try:
    print("\n📥 در حال نصب کتابخانه‌ها...")
    subprocess.run(["pip", "install", "-q", "audio-separator[gpu]==0.36.1"], check=True)
    subprocess.run(["pip", "install", "-q", "demucs"], check=True)
    subprocess.run(["pip", "install", "-q", "aria2"], check=True)
    subprocess.run(["pip", "install", "-q", "yt_dlp"], check=True)
    subprocess.run(["pip", "install", "-q", "pydub"], check=True)
    subprocess.run(["pip", "install", "-q", "tqdm"], check=True)

    print("📂 در حال ایجاد پوشه‌ها...")
    os.makedirs("models", exist_ok=True)
    os.makedirs("temp", exist_ok=True)

    print("⬇️ در حال دانلود مدل DrumSep...")
    result = subprocess.run([
        "aria2c",
        "https://huggingface.co/Eddycrack864/Drumsep/resolve/main/modelo_final.th",
        "-o", "models/drumsep.th",
        "--quiet=true"
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print("⚠️ هشدار: دانلود مدل DrumSep با مشکل مواجه شد")

    print("🔧 در حال نصب ابزارهای سیستمی...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "libcudnn8", "ffmpeg"], check=True)

    clear_output()
    print("✅ نصب با موفقیت انجام شد!")
    print("🚀 آماده برای پردازش")

except subprocess.CalledProcessError as e:
    print(f"❌ خطا در نصب: {str(e)}")
    raise
except Exception as e:
    print(f"❌ خطای غیرمنتظره: {str(e)}")
    raise

In [ ]:
#@title 🎵 **جداسازی صدا (Audio Separation)**

import os
import glob
import subprocess
import yt_dlp
import zipfile
from google.colab import files
from pydub import AudioSegment, silence
from tqdm import tqdm
from IPython.display import clear_output

#@markdown ### 🎯 **انتخاب هدف کاری**
User_Goal = "1. ساخت مدل هوش مصنوعی (وکال کریستالی)" #@param ["1. ساخت مدل هوش مصنوعی (وکال کریستالی)", "2. بیت‌ساز و ریمیکسر (تفکیک 4 لاین)", "3. نوازنده (گیتار و پیانو)", "4. یوتیوبر/تولید محتوا (حذف نویز و اکو)", "5. دی‌جی/کارائوکه (موزیک بی‌کلام قوی)"]

#@markdown ---
#@markdown ### 📂 **انتخاب فایل ورودی**
Upload_From_Computer = False #@param {type:"boolean"}
#@markdown <small>برای آپلود از حافظه داخلی، این گزینه را فعال کنید</small>

Audio_Input = "" #@param {type:"string"}
#@markdown <small>مسیر فایل در درایو یا لینک یوتیوب/صوت</small>

#@markdown ---
#@markdown ### 💾 **تنظیمات خروجی**
Output_Folder = "/content/drive/MyDrive/UVR5_Outputs" #@param {type:"string"}
Output_Format = "wav" #@param ["wav", "flac", "mp3"]
Download_As_Zip = False #@param {type:"boolean"}
#@markdown <small>فعال‌سازی دانلود مستقیم به صورت فایل زیپ</small>

#@markdown ---
#@markdown ### ✂️ **پردازش نهایی**
Remove_Silence = False #@param {type:"boolean"}
#@markdown <small>حذف سکوت‌های طولانی از فایل وکال (مناسب برای دیتاست)</small>

#@markdown ---
#@markdown ### ⚙️ **تنظیمات دستی (اختیاری)**
Manual_Override = False #@param {type:"boolean"}
Model_Select = "UVR-DeEcho-DeReverb.pth" #@param ["BS-Roformer-Viperx-1297", "MDX23C-8KFFT-InstVoc_HQ", "htdemucs_ft.yaml", "htdemucs_6s.yaml", "UVR-DeEcho-DeReverb.pth", "UVR-MDX-NET-Inst_HQ_5.onnx", "Mel-Band-Roformer-Kim"]

# ─────────────────────────────────────────────────────────────
# تعریف مدل‌ها و توضیحات
# ─────────────────────────────────────────────────────────────

MODEL_KZ = {
    "BS-Roformer-Viperx-1297": {
        "file": "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
        "arch": "mdxc",
        "seg": 256,
        "overlap": 8,
        "desc": "🎤 بهترین کیفیت برای جداسازی وکال - مناسب ساخت دیتاست AI"
    },
    "Mel-Band-Roformer-Kim": {
        "file": "mel_band_roformer_kim_ft2_bleedless_unwa.ckpt",
        "arch": "mdxc",
        "seg": 256,
        "overlap": 8,
        "desc": "🎵 وکال تمیز بدون نشت صدای موسیقی - مناسب کارهای حرفه‌ای"
    },
    "MDX23C-8KFFT-InstVoc_HQ": {
        "file": "MDX23C-8KFFT-InstVoc_HQ.ckpt",
        "arch": "mdxc",
        "seg": 256,
        "overlap": 8,
        "desc": "🎹 جداسازی وکال/اینسترومنتال با کیفیت بالا - مناسب دی‌جی و کارائوکه"
    },
    "htdemucs_ft.yaml": {
        "file": "htdemucs_ft.yaml",
        "arch": "demucs",
        "shifts": 2,
        "overlap": 0.25,
        "desc": "🎸 تفکیک 4 ساز (درامز، بیس، وکال، سایر) - بهینه برای ریمیکس و بیت‌سازی"
    },
    "htdemucs_6s.yaml": {
        "file": "htdemucs_6s.yaml",
        "arch": "demucs",
        "shifts": 2,
        "overlap": 0.25,
        "desc": "🎹 تفکیک 6 ساز (درامز، بیس، وکال، پیانو، گیتار، سایر) - مناسب نوازندگان"
    },
    "UVR-DeEcho-DeReverb.pth": {
        "file": "UVR-DeEcho-DeReverb.pth",
        "arch": "vr",
        "window": 512,
        "aggr": 5,
        "desc": "🎙️ حذف اکو و ریورب از وکال - ایده‌آل برای یوتیوبرها و پادکست"
    },
    "UVR-MDX-NET-Inst_HQ_5.onnx": {
        "file": "UVR-MDX-NET-Inst_HQ_5.onnx",
        "arch": "mdx",
        "seg": 256,
        "overlap": 0.25,
        "desc": "🎼 جداسازی سریع و با کیفیت وکال/موسیقی - تعادل بین سرعت و کیفیت"
    }
}

# ─────────────────────────────────────────────────────────────
# توابع کمکی
# ─────────────────────────────────────────────────────────────

def validate_inputs():
    """بررسی اعتبار ورودی‌های کاربر"""
    errors = []

    if not Upload_From_Computer and not Audio_Input:
        errors.append("❌ لطفاً یک فایل آپلود کنید یا مسیر/لینک فایل را وارد کنید")

    if Audio_Input and not Upload_From_Computer:
        if "http" not in Audio_Input and not os.path.exists(Audio_Input):
            errors.append(f"❌ فایل در مسیر '{Audio_Input}' یافت نشد")

    if Output_Format not in ["wav", "flac", "mp3"]:
        errors.append(f"❌ فرمت '{Output_Format}' پشتیبانی نمی‌شود")

    return errors

def get_settings_by_goal(goal):
    """انتخاب مدل بر اساس هدف کاربر"""
    goal_map = {
        "1.": "BS-Roformer-Viperx-1297",
        "2.": "htdemucs_ft.yaml",
        "3.": "htdemucs_6s.yaml",
        "4.": "UVR-DeEcho-DeReverb.pth",
        "5.": "MDX23C-8KFFT-InstVoc_HQ"
    }

    for key, model in goal_map.items():
        if key in goal:
            return MODEL_KZ[model]

    return MODEL_KZ["BS-Roformer-Viperx-1297"]

def downloader(url):
    """دانلود صوت از یوتیوب یا سایر منابع"""
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(f'{path}/temp', '%(title)s.%(ext)s'),
        'quiet': True,
        'no_warnings': True
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return True
    except Exception as e:
        print(f"❌ خطا در دانلود: {str(e)}")
        return False

def process_silence_removal(folder_path, format_ext):
    """حذف سکوت از فایل‌های وکال"""
    print(f"\n✂️ در حال پردازش حذف سکوت...")

    all_output_files = glob.glob(f"{folder_path}/*.{format_ext}")
    output_files_to_process = [f for f in all_output_files if "_NoSilence." not in os.path.basename(f)]

    processed_count = 0
    vocal_files = [f for f in output_files_to_process if "vocal" in os.path.basename(f).lower()]

    if not vocal_files:
        print("ℹ️ هیچ فایل وکالی برای حذف سکوت یافت نشد")
        return

    for file_path in tqdm(vocal_files, desc="حذف سکوت", unit="فایل"):
        filename = os.path.basename(file_path)

        try:
            audio = AudioSegment.from_file(file_path)

            chunks = silence.split_on_silence(
                audio,
                min_silence_len=500,
                silence_thresh=-45,
                keep_silence=100
            )

            if chunks:
                output_audio = chunks[0]
                for chunk in chunks[1:]:
                    output_audio += chunk

                base_name_without_ext, ext = os.path.splitext(filename)
                new_filename = f"{base_name_without_ext}_NoSilence{ext}"
                new_path = os.path.join(folder_path, new_filename)

                output_audio.export(new_path, format=format_ext)
                processed_count += 1
            else:
                print(f"⚠️ {filename}: فایل تقریباً خالی از صدا بود")

        except Exception as e:
            print(f"❌ خطا در پردازش {filename}: {str(e)}")

    if processed_count > 0:
        print(f"✅ حذف سکوت برای {processed_count} فایل انجام شد")

def create_zip_download(folder_path, format_ext):
    """ایجاد فایل ZIP و دانلود"""
    print("\n📦 در حال ایجاد فایل ZIP...")

    zip_filename = f"UVR5_Output_{format_ext}.zip"
    zip_path = os.path.join(path, zip_filename)

    output_files = glob.glob(f"{folder_path}/*.{format_ext}")

    if not output_files:
        print("⚠️ هیچ فایلی برای زیپ کردن یافت نشد")
        return

    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for file in tqdm(output_files, desc="زیپ کردن", unit="فایل"):
                zipf.write(file, os.path.basename(file))

        print(f"✅ فایل ZIP آماده دانلود است")
        files.download(zip_path)
        os.remove(zip_path)

    except Exception as e:
        print(f"❌ خطا در ایجاد ZIP: {str(e)}")

# ─────────────────────────────────────────────────────────────
# تابع اصلی پردازش
# ─────────────────────────────────────────────────────────────

def run_separation():
    """اجرای فرآیند جداسازی صدا"""

    # بررسی اعتبار ورودی‌ها
    validation_errors = validate_inputs()
    if validation_errors:
        for error in validation_errors:
            print(error)
        return

    # تعیین تنظیمات
    if Manual_Override:
        current_settings = MODEL_KZ[Model_Select]
        print(f"🔧 حالت دستی: {Model_Select}")
        print(f"ℹ️  {current_settings['desc']}")
    else:
        current_settings = get_settings_by_goal(User_Goal)
        selected_model = [k for k, v in MODEL_KZ.items() if v == current_settings][0]
        print(f"🎯 بهینه‌سازی خودکار برای: {User_Goal}")
        print(f"📌 مدل انتخابی: {selected_model}")
        print(f"ℹ️  {current_settings['desc']}")

    model_file = current_settings["file"]
    arch = current_settings["arch"]

    # پاکسازی پوشه temp
    if os.path.exists(f"{path}/temp"):
        for f in glob.glob(f"{path}/temp/*"):
            try:
                os.remove(f)
            except:
                pass
    else:
        os.makedirs(f"{path}/temp")

    # آماده‌سازی ورودی
    input_path = Audio_Input

    if Upload_From_Computer:
        print("\n⬆️ لطفاً فایل خود را آپلود کنید...")
        uploaded = files.upload()

        if not uploaded:
            print("❌ فایلی آپلود نشد")
            return

        for filename in uploaded.keys():
            src = os.path.join(os.getcwd(), filename)
            dst = os.path.join(f"{path}/temp", filename)
            if os.path.exists(src):
                os.rename(src, dst)

        input_path = f"{path}/temp"
        print("✅ آپلود موفق")

    elif "http" in Audio_Input:
        print("\n⬇️ در حال دانلود...")
        if not downloader(Audio_Input):
            return
        input_path = f"{path}/temp"
        print("✅ دانلود موفق")

    # جستجوی فایل‌های صوتی
    found_files = []
    extensions = (".wav", ".flac", ".mp3", ".ogg", ".m4a")

    if os.path.isdir(input_path):
        for f in os.listdir(input_path):
            if f.lower().endswith(extensions):
                found_files.append(os.path.join(input_path, f))
    else:
        if input_path.lower().endswith(extensions):
            found_files.append(input_path)

    if not found_files:
        print("❌ هیچ فایل صوتی قابل پردازشی یافت نشد")
        return

    print(f"\n🚀 شروع پردازش {len(found_files)} فایل...")

    # ایجاد پوشه خروجی
    os.makedirs(Output_Folder, exist_ok=True)

    # پردازش فایل‌ها
    for idx, file_path in enumerate(found_files, 1):
        print(f"\n{'='*60}")
        print(f"📁 فایل {idx}/{len(found_files)}: {os.path.basename(file_path)}")
        print(f"{'='*60}")

        base_cmd = f'audio-separator "{file_path}" --model_filename {model_file} --output_dir="{Output_Folder}" --output_format={Output_Format} --model_file_dir=./models --use_autocast'

        if arch == "mdxc":
            cmd = f'{base_cmd} --mdxc_segment_size={current_settings["seg"]} --mdxc_overlap={current_settings["overlap"]} --mdxc_batch_size=1'
        elif arch == "demucs":
            cmd = f'{base_cmd} --demucs_shifts={current_settings["shifts"]} --demucs_overlap={current_settings["overlap"]}'
        elif arch == "vr":
            cmd = f'{base_cmd} --vr_window_size={current_settings["window"]} --vr_aggression={current_settings["aggr"]} --vr_batch_size=1'
        elif arch == "mdx":
            cmd = f'{base_cmd} --mdx_segment_size={current_settings["seg"]} --mdx_overlap={current_settings["overlap"]} --mdx_batch_size=1'

        try:
            subprocess.run(cmd, shell=True, check=True)
            print(f"✅ پردازش فایل {idx} تکمیل شد")
        except subprocess.CalledProcessError as e:
            print(f"❌ خطا در پردازش فایل {idx}: {str(e)}")

    # حذف سکوت
    if Remove_Silence:
        process_silence_removal(Output_Folder, Output_Format)

    # دانلود ZIP
    if Download_As_Zip:
        create_zip_download(Output_Folder, Output_Format)

    # پاکسازی
    if "http" in Audio_Input or Upload_From_Computer:
        for f in glob.glob(f"{path}/temp/*"):
            try:
                os.remove(f)
            except:
                pass

    print("\n" + "="*60)
    print("🎉 جداسازی صدا با موفقیت انجام شد!")
    if not Download_As_Zip:
        print(f"📂 فایل‌های خروجی در: {Output_Folder}")
    print("="*60)

# اجرای برنامه
try:
    run_separation()
except Exception as e:
    print(f"\n❌ خطای کلی: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:

#@title 📂 **مدیریت هوشمند فایل (نسخه موبایل و دسکتاپ)**

import os
import html
from IPython.display import HTML, display

#@markdown مسیر پوشه خروجی را وارد کنید:
Target_Folder = "/content/drive/MyDrive/UVR5_Outputs" #@param {type:"string"}

def list_refined_files_v2(folder_path):
    if not os.path.exists(folder_path):
        display(HTML(f"<div style='color:red; padding:10px;'>❌ مسیر یافت نشد: {folder_path}</div>"))
        return

    files_found = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.wav', '.mp3', '.flac')):
                if "Instrumental" in file: continue
                if "(Reverb)" in file and "(No Reverb)" not in file: continue

                full_path = os.path.join(root, file)
                files_found.append((file, full_path))

    if not files_found:
        print("⚠️ فایلی پیدا نشد.")
        return

    # مرتب‌سازی: No Reverb بالاتر از همه
    files_found.sort(key=lambda x: "No Reverb" in x[0], reverse=True)

    html_code = """
    <style>
        .container { font-family: sans-serif; direction: rtl; background: #1a1a1a; padding: 10px; border-radius: 10px; color: white; }
        .file-card {
            background: #252525;
            border-radius: 8px;
            padding: 15px;
            margin-bottom: 15px;
            border-right: 4px solid #00d2ff;
            display: flex;
            flex-direction: column; /* چیدمان عمودی برای موبایل */
            gap: 10px;
        }
        .tag {
            display: inline-block;
            padding: 4px 10px;
            border-radius: 5px;
            font-size: 12px;
            font-weight: bold;
            margin-bottom: 5px;
        }
        .tag-crystal { background: #00d2ff; color: #000; }
        .tag-vocal { background: #4caf50; color: white; }

        .file-name {
            font-size: 14px;
            word-break: break-all; /* شکستن کلمات طولانی */
            direction: ltr;
            text-align: left;
            color: #efefef;
            line-height: 1.4;
        }
        .copy-btn {
            background: #007bff;
            color: white;
            border: none;
            padding: 12px;
            border-radius: 6px;
            cursor: pointer;
            font-size: 14px;
            font-weight: bold;
            width: 100%; /* دکمه تمام عرض برای راحتی لمس در موبایل */
            margin-top: 5px;
        }
    </style>
    <script>
        function copyText(text, btnId) {
            var textArea = document.createElement("textarea");
            textArea.value = text;
            document.body.appendChild(textArea);
            textArea.select();
            try {
                document.execCommand('copy');
                var btn = document.getElementById(btnId);
                btn.innerText = "✅ کپی شد!";
                btn.style.background = "#28a745";
                setTimeout(function(){
                    btn.innerText = "📋 کپی مسیر فایل";
                    btn.style.background = "#007bff";
                }, 2000);
            } catch (err) {
                alert('خطا در کپی');
            }
            document.body.removeChild(textArea);
        }
    </script>
    <div class="container">
        <h3 style="text-align: center; color: #00d2ff;">✨ لیست فایل‌های نهایی</h3>
    """

    for i, (name, full_path) in enumerate(files_found):
        btn_id = f"btn_new_{i}"

        # انتخاب تگ
        if "No Reverb" in name:
            tag = '<span class="tag tag-crystal">💎 Crystal Clear</span>'
            border = "border-right: 4px solid #00d2ff;"
        elif "NoSilence" in name:
            tag = '<span class="tag tag-vocal">🎙️ No Silence</span>'
            border = "border-right: 4px solid #4caf50;"
        else:
            tag = '<span class="tag" style="background:#666;">🎵 Raw</span>'
            border = "border-right: 4px solid #888;"

        html_code += f"""
        <div class="file-card" style="{border}">
            <div>{tag}</div>
            <div class="file-name">{name}</div>
            <button id="{btn_id}" class="copy-btn" onclick="copyText('{html.escape(full_path)}', '{btn_id}')">📋 کپی مسیر فایل</button>
        </div>
        """

    html_code += "</div>"
    display(HTML(html_code))

list_refined_files_v2(Target_Folder)

In [ ]:

#@title 🔊 **نرمالایز تخصصی وکال کریستالی (Crystal Normalize)**

import os
import glob
from pydub import AudioSegment
from pydub.effects import normalize
from tqdm import tqdm

Input_Folder = "/content/drive/MyDrive/UVR5_Outputs" #@param {type:"string"}
Normalization_Target = "peak" #@param ["peak", "lufs"]
Target_dBFS = -3.0 #@param {type:"number"}
Download_Normalized_As_Zip = False #@param {type:"boolean"}

def check_crystal_vocals():
    if not os.path.exists(Input_Folder):
        return False, []

    extensions = ('*.wav', '*.flac', '*.mp3')
    all_files = []
    for ext in extensions:
        all_files.extend(glob.glob(os.path.join(Input_Folder, ext)))

    crystal_files = []
    for f in all_files:
        filename = os.path.basename(f)
        if "NoSilence" in filename and "(No Reverb)" in filename:
            if "_Normalized" not in filename:
                crystal_files.append(f)
    return len(crystal_files) > 0, crystal_files

def run_crystal_normalization():
    print("🔍 در حال جستجوی فایل‌های واجد شرایط...")
    has_files, target_files = check_crystal_vocals()

    if not has_files:
        print("⚠️ فایلی با مشخصات NoSilence + No Reverb پیدا نشد.")
        return

    success_files = []
    for file_path in tqdm(target_files, desc="Processing"):
        try:
            audio = AudioSegment.from_file(file_path)

            if Normalization_Target == "peak":
                audio_norm = normalize(audio)
                diff = Target_dBFS - audio_norm.max_dBFS
                final_audio = audio_norm + diff
            else:
                diff = Target_dBFS - audio.dBFS
                final_audio = audio + diff

            base, ext = os.path.splitext(file_path)
            new_path = f"{base}_Normalized{ext}"

            # اجبار به استفاده از فرمت صحیح
            fmt = ext.replace('.', '').lower()
            final_audio.export(new_path, format=fmt)

            if os.path.exists(new_path):
                success_files.append(new_path)
        except Exception as e:
            print(f"❌ Error in {os.path.basename(file_path)}: {e}")

    print(f"\n{'='*50}")
    if success_files:
        print(f"✅ موفقیت‌آمیز: {len(success_files)} فایل نرمالایز شد.")
        for p in success_files: print(f"📍 Created: {os.path.basename(p)}")
    else:
        print("❌ هیچ فایلی ذخیره نشد. دسترسی به درایو را چک کنید.")
    print(f"{'='*50}")

    if Download_Normalized_As_Zip and success_files:
        import zipfile
        from google.colab import files as colab_files
        z_path = "/content/Normalized_Result.zip"
        with zipfile.ZipFile(z_path, 'w', zipfile.ZIP_DEFLATED) as z:
            for f in success_files: z.write(f, os.path.basename(f))
        colab_files.download(z_path)

run_crystal_normalization()

###برای نرمالایز صدا توسط اپلیکیشن ، این ویدیو را تماشا کنید
[![آموزش ویدیویی](https://img.youtube.com/vi/vxi_7frcXhY/0.jpg)](https://youtu.be/vxi_7frcXhY?si=wP5xfoHvw1tU28H2)

[🎥 مشاهده ویدیو در یوتیوب](https://youtu.be/vxi_7frcXhY?si=LngLfdr7sAeekA6Y)